# BTC Volume Pattern Checker (Google Colab)

**Instrukcja:** Zmień TIMEFRAME poniżej na dowolny z listy (od 1m do 1M). Uruchom komórkę i zobacz wynik. Działa bez hasła i emaila.

In [ ]:
# === USTAWIENIA ===
TIMEFRAME = "1m"   # dostępne: 1m, 3m, 5m, 15m, 30m, 1h, 2h, 4h, 6h, 12h, 1d, 3d, 1w, 1M

import requests
from datetime import datetime

GRANULARITY_MAP = {
    "1m": 60, "3m": 180, "5m": 300, "15m": 900, "30m": 1800,
    "1h": 3600, "2h": 7200, "4h": 14400, "6h": 21600, "12h": 43200,
    "1d": 86400, "3d": 259200, "1w": 604800, "1M": 2592000
}

granularity = GRANULARITY_MAP.get(TIMEFRAME)
if granularity is None:
    print("Nieobsługiwany timeframe! Wybierz z listy.")
    granularity = 60

def get_bars():
    url = f"https://api.exchange.coinbase.com/products/BTC-USD/candles?granularity={granularity}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if not isinstance(data, list) or len(data) == 0:
            print("Błąd API:", data)
            return []
        data = data[-10:]
        bars = []
        for bar in data:
            open_time = datetime.fromtimestamp(bar[0]).strftime('%H:%M')
            o = float(bar[3])
            c = float(bar[4])
            v = float(bar[5])
            color = "green" if c > o else "red"
            bars.append({"time": open_time, "open": o, "close": c, "volume": v, "color": color})
        return bars
    except Exception as e:
        print("Błąd pobierania:", e)
        return []

def check_volume_pattern(bars):
    if len(bars) < 2: return "Za mało danych."
    for i in range(1, len(bars)):
        curr = bars[i]
        prev = bars[i-1]
        if curr["color"] != prev["color"]:
            last_vol = next((bars[j]["volume"] for j in range(i-1, -1, -1) if bars[j]["color"] == curr["color"]), 0)
            if curr["volume"] > last_vol:
                return f"WZORZEC WYKRYTY! {curr['time']} {curr['color']} vol={curr['volume']:.2f} > prev {last_vol:.2f}"
    return "Brak wzorca w ostatnich 10 barach."

print(f"=== BTC Volume Pattern - {TIMEFRAME} ===")
bars = get_bars()
if bars:
    for b in bars: print(f"{b['time']} | {b['color']:6} | vol {b['volume']:.2f}")
    print(check_volume_pattern(bars))
else:
    print("Brak danych.")